# 📊 Step 2 — Dataset Understanding & Exploratory Data Analysis (EDA)

**Project:** Customer Churn ML Prediction Pipeline  
**Dataset:** Telco Customer Churn (CSV)  
**Goal:** Understand the raw data deeply *before* writing any preprocessing code.

---

### Why EDA matters
- Tells you which columns need cleaning and how.
- Reveals class imbalance → influences model choice and evaluation metrics.
- Exposes data quality issues that would silently corrupt your model.
- Gives you talking points in interviews ("I noticed X, so I did Y").

### Notebook sections
1. Load the dataset
2. Inspect shape and column names
3. Detect missing values
4. Summarise numerical columns
5. Summarise categorical columns
6. Inspect class imbalance (churn target)
7. Key insights and recommended next steps

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Make plots render inline and look clean
%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 4)

print("Libraries loaded ✓")

---
## 1. Load the Dataset

We load the raw CSV using `pd.read_csv()` and immediately display the first few rows.

> **What to look for:** Obvious noise in column names (spaces, mixed case), unexpected data types, and whether the target column `Churn` is present.

In [ ]:
# ── Load CSV ───────────────────────────────────────────────────────────────
# Adjust this path if your file has a different name
DATA_PATH = Path("../data/raw/churn.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}.\n"
        "Download from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn\n"
        "Then place the file at: data/raw/churn.csv"
    )

df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded successfully — {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

---
## 2. Inspect Shape and Column Names

Understanding the dimensions and column list is the very first sanity check.  
It tells you: how many data points you have, how many features, and whether expected columns are present.

In [ ]:
# ── Shape ──────────────────────────────────────────────────────────────────
rows, cols = df.shape
print(f"Rows    : {rows:,}")
print(f"Columns : {cols}")

# ── Column names with their data types ─────────────────────────────────────
print("\n── Column Names & Data Types ──")
col_info = pd.DataFrame({
    "column":    df.columns,
    "dtype":     df.dtypes.values,
    "non_null":  df.notnull().sum().values,
    "null_count": df.isnull().sum().values,
})
col_info

---
## 3. Detect Missing Values

A common source of bugs: if you don't handle nulls explicitly, sklearn will crash or silently produce bad predictions.

> **What to look for:**
> - Any column with > 50% missing → consider dropping entirely.
> - Numeric columns with a few nulls → fill with median (safer than mean for skewed data).
> - Categorical columns with a few nulls → fill with mode or a constant `"Unknown"`.
> - `TotalCharges` in the Telco dataset is stored as a **string with whitespace** — that's a data type bug, not a true missing value.

In [ ]:
# ── Count missing values ───────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "missing_count": missing,
    "missing_%": missing_pct
}).sort_values("missing_%", ascending=False)

print("Columns with at least one null:")
print(missing_df[missing_df["missing_count"] > 0].to_string())
print(f"\nTotal null values in entire DataFrame: {df.isnull().sum().sum()}")

# ── Visual: missing value bar chart ───────────────────────────────────────
# Only plot columns that actually have nulls
cols_with_nulls = missing_df[missing_df["missing_count"] > 0]

if cols_with_nulls.empty:
    print("\nNo missing values found — the raw CSV looks complete.")
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(
        x=cols_with_nulls.index,
        y=cols_with_nulls["missing_%"],
        palette="Reds_r",
        ax=ax,
    )
    ax.set_title("Missing Value Percentage by Column")
    ax.set_xlabel("Column")
    ax.set_ylabel("Missing (%)")
    ax.axhline(50, color="red", linestyle="--", label="50% threshold")
    ax.legend()
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

# ── Special check: TotalCharges whitespace bug ─────────────────────────────
# In the Telco dataset, TotalCharges has " " (space) rows → hidden nulls
if "TotalCharges" in df.columns:
    whitespace_rows = (df["TotalCharges"].astype(str).str.strip() == "").sum()
    print(f"\n⚠  'TotalCharges' rows with whitespace-only value: {whitespace_rows}")
    print("   These will become NaN when cast to float — handle in preprocessing.")

---
## 4. Summarise Numerical Columns

`df.describe()` gives you mean, std, min, max, and quartiles in one call.  
Pair it with histograms and boxplots to spot skewness and outliers.

> **What to look for:**
> - **Skewed distributions** (e.g. `TotalCharges`) → StandardScaler handles this reasonably; log-transform is optional.
> - **Wide value ranges** (e.g. `tenure` 0–72 vs `MonthlyCharges` 0–118) → StandardScaler is important so no feature dominates.
> - **Outliers** in boxplots → decide whether to cap (Winsorize) or leave as-is.
> - **`SeniorCitizen`** is stored as 0/1 int but is actually categorical — rename/treat accordingly.

In [ ]:
# ── Descriptive statistics for numeric columns ─────────────────────────────
# Convert TotalCharges to numeric first if it exists (it arrives as object)
df_num = df.copy()
if "TotalCharges" in df_num.columns:
    df_num["TotalCharges"] = pd.to_numeric(df_num["TotalCharges"], errors="coerce")

numeric_cols = df_num.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}\n")

df_num[numeric_cols].describe().round(2)

In [ ]:
# ── Histograms — distribution of each numeric feature ─────────────────────
# Exclude SeniorCitizen (binary 0/1 — better treated as categorical)
plot_numeric = [c for c in numeric_cols if c.lower() != "seniorcitizen"]

fig, axes = plt.subplots(1, len(plot_numeric), figsize=(5 * len(plot_numeric), 4))
if len(plot_numeric) == 1:
    axes = [axes]

for ax, col in zip(axes, plot_numeric):
    df_num[col].dropna().plot.hist(bins=30, ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(col)
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")
    # Annotate skewness
    skew = df_num[col].skew()
    ax.text(
        0.97, 0.95, f"skew={skew:.2f}",
        ha="right", va="top", transform=ax.transAxes,
        fontsize=9, color="darkred"
    )

plt.suptitle("Numerical Feature Distributions", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Boxplots — outliers split by Churn label ──────────────────────────────
# Seeing how numeric features differ between churned vs retained customers
# is one of the most useful early signals.

target_col = "Churn"   # adjust if your dataset uses a different name

if target_col in df_num.columns:
    fig, axes = plt.subplots(1, len(plot_numeric), figsize=(5 * len(plot_numeric), 5))
    if len(plot_numeric) == 1:
        axes = [axes]

    for ax, col in zip(axes, plot_numeric):
        sns.boxplot(
            data=df_num,
            x=target_col,
            y=col,
            palette={"Yes": "#e74c3c", "No": "#2ecc71"},
            ax=ax,
        )
        ax.set_title(f"{col} by Churn")
        ax.set_xlabel("Churn")

    plt.suptitle("Numeric Features vs Churn (Outlier View)", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print(f"Target column '{target_col}' not found — skipping boxplot.")

---
## 5. Summarise Categorical Columns

Categorical features need encoding before the model can use them.
Knowing their unique values now helps you choose the right encoding strategy.

> **What to look for:**
> - **High cardinality** (many unique values, e.g. `customerID`) → usually dropped, not encoded.
> - **Binary columns** (`Yes`/`No`) → simple Label Encoding (0/1) is sufficient.
> - **Multi-value columns** (e.g. `Contract`: Month-to-month / One year / Two year) → One-Hot Encoding.
> - **Columns with rare categories** (< ~1% frequency) → consider grouping into `"Other"` to avoid overfitting.

In [ ]:
# ── Categorical column overview ────────────────────────────────────────────
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

# Remove the target column from feature list for this section
cat_feature_cols = [c for c in cat_cols if c.lower() != target_col.lower()]

print(f"Categorical feature columns ({len(cat_feature_cols)}):\n")

cat_summary = pd.DataFrame({
    "column":      cat_feature_cols,
    "unique_vals": [df[c].nunique() for c in cat_feature_cols],
    "top_value":   [df[c].mode()[0] if not df[c].mode().empty else "N/A" for c in cat_feature_cols],
    "top_freq_%":  [(df[c].value_counts(normalize=True).iloc[0] * 100).round(1) for c in cat_feature_cols],
}).sort_values("unique_vals", ascending=False)

print(cat_summary.to_string(index=False))

In [ ]:
# ── Count plots for each categorical feature (vs Churn) ───────────────────
# Only plot columns with a small number of categories (≤ 6) to keep plots readable
plot_cat_cols = [c for c in cat_feature_cols if df[c].nunique() <= 6]

if plot_cat_cols:
    n_cols = 3
    n_rows = (len(plot_cat_cols) + n_cols - 1) // n_cols   # ceiling division
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
    axes = axes.flatten()

    for i, col in enumerate(plot_cat_cols):
        if target_col in df.columns:
            sns.countplot(
                data=df,
                x=col,
                hue=target_col,
                palette={"Yes": "#e74c3c", "No": "#2ecc71"},
                ax=axes[i],
            )
            axes[i].legend(title="Churn", labels=["No", "Yes"])
        else:
            sns.countplot(data=df, x=col, ax=axes[i], color="steelblue")

        axes[i].set_title(col)
        axes[i].set_xlabel("")
        axes[i].tick_params(axis="x", rotation=30)

    # Hide any unused subplots
    for j in range(len(plot_cat_cols), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("Categorical Features vs Churn", fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("No categorical columns with ≤ 6 unique values to plot.")

---
## 6. Inspect Class Imbalance (Churn Target)

Class imbalance is **the most important EDA finding** for churn prediction.

In a typical telecom dataset, ~73% of customers stay and ~27% churn.  
This means a "dumb" model that predicts "No Churn" for everyone would achieve 73% accuracy — which is misleading.

> **Implications:**
> - Don't optimise for **accuracy** — use **ROC-AUC** and **Recall** instead.
> - Use `class_weight="balanced"` in sklearn models to penalise the majority class less.
> - If imbalance is severe (> 90/10), consider SMOTE oversampling (from `imbalanced-learn`).
> - Always use **stratified** train/test split (`stratify=y`) so both sets have the same churn ratio.

In [ ]:
# ── Class distribution — counts and percentages ────────────────────────────
if target_col not in df.columns:
    print(f"Target column '{target_col}' not found. Update the target_col variable.")
else:
    counts = df[target_col].value_counts()
    pcts   = df[target_col].value_counts(normalize=True).mul(100).round(2)

    class_df = pd.DataFrame({"count": counts, "percentage_%": pcts})
    print("── Churn Class Distribution ──")
    print(class_df.to_string())

    imbalance_ratio = counts.max() / counts.min()
    print(f"\nImbalance ratio (majority : minority) = {imbalance_ratio:.1f} : 1")

    if imbalance_ratio < 3:
        print("✓  Mild imbalance — class_weight='balanced' is sufficient.")
    elif imbalance_ratio < 10:
        print("⚠  Moderate imbalance — use class_weight='balanced' and monitor Recall.")
    else:
        print("✗  Severe imbalance — consider SMOTE or other resampling techniques.")

In [ ]:
# ── Class imbalance bar chart ──────────────────────────────────────────────
if target_col in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    # Left: raw counts
    colours = ["#2ecc71", "#e74c3c"]
    counts.plot.bar(ax=axes[0], color=colours, edgecolor="white", width=0.5)
    axes[0].set_title("Churn — Raw Counts")
    axes[0].set_xlabel("Churn")
    axes[0].set_ylabel("Number of customers")
    axes[0].tick_params(axis="x", rotation=0)
    for bar in axes[0].patches:
        axes[0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 30,
            f"{int(bar.get_height()):,}",
            ha="center", va="bottom", fontsize=10
        )

    # Right: percentage pie chart
    axes[1].pie(
        counts,
        labels=[f"{l}\n({p:.1f}%)" for l, p in zip(counts.index, pcts)],
        colors=colours,
        startangle=140,
        wedgeprops={"edgecolor": "white", "linewidth": 2},
        textprops={"fontsize": 11},
    )
    axes[1].set_title("Churn — Class Split (%)")

    plt.suptitle("Class Imbalance Analysis — Churn Target", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

---
## 7. Key Insights to Look For — Checklist Before Preprocessing

Run this as a final summary of your findings.

In [ ]:
# ── Correlation heatmap (numeric features only) ────────────────────────────
# High correlation between two features (> 0.8) → one may be redundant.
# E.g. tenure and TotalCharges are likely highly correlated.

if len(numeric_cols) > 1:
    corr = df_num[numeric_cols].corr()

    fig, ax = plt.subplots(figsize=(7, 5))
    mask = np.triu(np.ones_like(corr, dtype=bool))   # hide upper triangle (duplicate)
    sns.heatmap(
        corr,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0,
        square=True,
        linewidths=0.5,
        ax=ax,
    )
    ax.set_title("Correlation Matrix — Numeric Features", fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric columns for a correlation matrix.")

In [ ]:
# ── Automated EDA Checklist ────────────────────────────────────────────────
# This cell prints a plain-English summary you can take straight into preprocessing.

print("=" * 60)
print("  EDA CHECKLIST SUMMARY")
print("=" * 60)

# 1. Missing values
total_missing = df.isnull().sum().sum()
high_missing_cols = missing_df[missing_df["missing_%"] > 20].index.tolist()
print(f"\n1. Missing Values")
print(f"   Total nulls          : {total_missing}")
print(f"   High-missing (>20%)  : {high_missing_cols if high_missing_cols else 'None'}")
if "TotalCharges" in df.columns:
    ws = (df["TotalCharges"].astype(str).str.strip() == "").sum()
    print(f"   TotalCharges whitespace rows : {ws}  → cast to float, will → NaN")

# 2. Numeric features
print(f"\n2. Numeric Features ({len(numeric_cols)})")
for col in numeric_cols:
    skew = df_num[col].skew()
    flag = "⚠ right-skewed" if skew > 1 else ("⚠ left-skewed" if skew < -1 else "✓ normal-ish")
    print(f"   {col:<20} skew={skew:+.2f}  {flag}")

# 3. Categorical features
print(f"\n3. Categorical Features ({len(cat_feature_cols)})")
for col in cat_feature_cols:
    n = df[col].nunique()
    flag = "⚠ high cardinality — consider dropping" if n > 20 else (
           "binary → label encode" if n == 2 else "→ one-hot encode"
    )
    print(f"   {col:<30} unique={n:>3}   {flag}")

# 4. Class imbalance
if target_col in df.columns:
    pct_churn = df[target_col].value_counts(normalize=True).get("Yes",
                df[target_col].value_counts(normalize=True).get(1, 0)) * 100
    print(f"\n4. Class Imbalance")
    print(f"   Churn rate : {pct_churn:.1f}%")
    if pct_churn < 30:
        print("   → Use class_weight='balanced' | Evaluate with ROC-AUC & Recall")
    else:
        print("   → Classes are reasonably balanced")

print("\n" + "=" * 60)
print("  RECOMMENDED NEXT STEPS FOR PREPROCESSING")
print("=" * 60)
print("""
  1. Standardise column names        → src/preprocessing.py
  2. Cast TotalCharges → float        → src/preprocessing.py
  3. Fill missing numerics (median)   → src/preprocessing.py
  4. Drop customerID (not a feature)  → src/feature_engineering.py
  5. One-hot encode multi-value cols  → src/feature_engineering.py
  6. Label encode binary Yes/No cols  → src/feature_engineering.py
  7. StandardScaler on numeric cols   → src/feature_engineering.py
  8. Stratified train/test split      → src/feature_engineering.py
""")

---

## ✅ EDA Complete — What You Now Know

| Finding | Implication |
|---|---|
| `TotalCharges` is a string with whitespace rows | Cast to float in preprocessing; whitespace → NaN → fill with median |
| `tenure`, `MonthlyCharges`, `TotalCharges` are right-skewed | StandardScaler handles this; log-transform is optional |
| `tenure` and `TotalCharges` are highly correlated | One carries most of the signal; consider dropping one if model is overfitting |
| `customerID` is high-cardinality, unique per row | Drop it — not a feature |
| ~15 binary Yes/No categorical columns | Label encode (0/1) |
| `Contract`, `PaymentMethod`, `InternetService` have 2–4 categories | One-Hot Encode |
| ~73% No Churn / ~27% Churn | Use `class_weight="balanced"` + evaluate with ROC-AUC & Recall |

---

**Next Step → `src/preprocessing.py` already handles all of the above.**  
Run Step 3 (preprocessing + feature engineering) when ready.